# 🔬 Laboratorio de Ciencia de Datos - Taller Analítica Massy Group\n\n**Objetivo:** Analizar datos de talleres para construir la estrategia analítica de Massy.\n\n**Capas:** Estratégico (C-Level) | Táctico (Líderes) | Operativo (Día a día)

In [ ]:
# Instalación de dependencias (ejecutar solo una vez)\n# !pip install pymssql pandas matplotlib seaborn openai python-dotenv openpyxl wordcloud\n\nimport pymssql\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport json, os\nfrom dotenv import load_dotenv\n\nload_dotenv(dotenv_path='../.env')\n\nplt.style.use('seaborn-v0_8-whitegrid')\nsns.set_palette('husl')\nplt.rcParams['figure.figsize'] = (12, 6)\nplt.rcParams['font.size'] = 11\nprint('✅ Librerías cargadas')

In [ ]:
# Conexión a SQL Server\nDB_SERVER = os.getenv('DB_SERVER', '162.248.53.192')\nDB_USER = os.getenv('DB_USER', 'IdentyWebUser')\nDB_PASSWORD = os.getenv('DB_PASSWORD', 'Tatiana2006')\nDB_NAME = os.getenv('DB_NAME', 'taller_analitica')\n\nconn = pymssql.connect(server=DB_SERVER, user=DB_USER, password=DB_PASSWORD, database=DB_NAME)\nprint(f'✅ Conectado a {DB_NAME} en {DB_SERVER}')\n\n# Cargar todas las tablas\ndf_gerentes = pd.read_sql('SELECT * FROM gerentes', conn)\ndf_decisiones = pd.read_sql('SELECT d.*, g.nombre, g.area, g.capa FROM decisiones d JOIN gerentes g ON d.gerente_id=g.id', conn)\ndf_preguntas = pd.read_sql('SELECT p.*, g.nombre, g.area, g.capa, d.decision, d.frecuencia, d.impacto FROM preguntas_criticas p JOIN gerentes g ON p.gerente_id=g.id LEFT JOIN decisiones d ON p.decision_id=d.id', conn)\ndf_fricciones = pd.read_sql('SELECT f.*, g.nombre, g.area, g.capa, p.pregunta_clave, d.decision FROM fricciones f JOIN gerentes g ON f.gerente_id=g.id LEFT JOIN preguntas_criticas p ON f.pregunta_critica_id=p.id LEFT JOIN decisiones d ON p.decision_id=d.id', conn)\ndf_votaciones = pd.read_sql('SELECT v.*, g.nombre, p.pregunta_clave FROM votaciones v JOIN gerentes g ON v.gerente_id=g.id JOIN preguntas_criticas p ON v.pregunta_critica_id=p.id', conn)\n\nprint(f'📊 Gerentes: {len(df_gerentes)} | Decisiones: {len(df_decisiones)} | Preguntas: {len(df_preguntas)} | Fricciones: {len(df_fricciones)} | Votos: {len(df_votaciones)}')

## 1. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Resumen general del taller\nfig, axes = plt.subplots(2, 2, figsize=(14, 10))\nfig.suptitle('Resumen General del Taller de Analítica - Massy Group', fontsize=16, fontweight='bold')\n\n# 1. Distribución por Impacto\nif len(df_decisiones) > 0:\n    impacto_order = ['Bajo', 'Medio', 'Alto', 'Crítico']\n    impacto_colors = {'Bajo': '#10b981', 'Medio': '#f59e0b', 'Alto': '#ef4444', 'Crítico': '#991b1b'}\n    impacto_counts = df_decisiones['impacto'].value_counts().reindex(impacto_order, fill_value=0)\n    axes[0,0].bar(impacto_counts.index, impacto_counts.values, color=[impacto_colors.get(x, '#94a3b8') for x in impacto_counts.index])\n    axes[0,0].set_title('Distribución por Impacto')\n    axes[0,0].set_ylabel('Cantidad')\n\n# 2. Distribución por Frecuencia\nif len(df_decisiones) > 0:\n    freq_order = ['Diaria', 'Semanal', 'Quincenal', 'Mensual', 'Trimestral', 'Anual']\n    freq_counts = df_decisiones['frecuencia'].value_counts().reindex(freq_order, fill_value=0)\n    axes[0,1].barh(freq_counts.index, freq_counts.values, color='#3b82f6')\n    axes[0,1].set_title('Distribución por Frecuencia')\n    axes[0,1].invert_yaxis()\n\n# 3. Participación por Área\nif len(df_gerentes) > 0:\n    area_counts = df_gerentes['area'].value_counts()\n    axes[1,0].pie(area_counts.values, labels=area_counts.index, autopct='%1.0f%%', startangle=90)\n    axes[1,0].set_title('Participantes por Área')\n\n# 4. Métricas por Capa\nif len(df_gerentes) > 0:\n    capa_data = df_gerentes['capa'].value_counts()\n    axes[1,1].bar(capa_data.index, capa_data.values, color=['#1e3a5f', '#3b82f6', '#93c5fd'])\n    axes[1,1].set_title('Participantes por Capa Organizacional')\n\nplt.tight_layout()\nplt.show()

## 2. Heatmap: Frecuencia × Impacto (ROI Analítico)

In [ ]:
# Heatmap Frecuencia x Impacto\nif len(df_decisiones) > 0:\n    freq_order = ['Diaria', 'Semanal', 'Quincenal', 'Mensual', 'Trimestral', 'Anual']\n    imp_order = ['Bajo', 'Medio', 'Alto', 'Crítico']\n    pivot = df_decisiones.pivot_table(index='frecuencia', columns='impacto', aggfunc='size', fill_value=0)\n    pivot = pivot.reindex(index=freq_order, columns=imp_order, fill_value=0)\n    \n    fig, ax = plt.subplots(figsize=(10, 6))\n    sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd', linewidths=1, ax=ax, cbar_kws={'label': 'Cantidad de Decisiones'})\n    ax.set_title('Heatmap: Frecuencia × Impacto\\n(Mayor ROI analítico = Alta Frecuencia + Alto Impacto)', fontsize=14, fontweight='bold')\n    ax.set_ylabel('Frecuencia de la Decisión')\n    ax.set_xlabel('Nivel de Impacto')\n    plt.tight_layout()\n    plt.show()\nelse:\n    print('⚠️ No hay decisiones para generar el heatmap')

## 3. Scoring Compuesto y Priorización de Preguntas Críticas

In [ ]:
# Scoring compuesto por pregunta crítica\nif len(df_preguntas) > 0:\n    freq_scores = {'Diaria': 6, 'Semanal': 5, 'Quincenal': 4, 'Mensual': 3, 'Trimestral': 2, 'Anual': 1}\n    imp_scores = {'Crítico': 8, 'Alto': 6, 'Medio': 3, 'Bajo': 1}\n    \n    # Contar votos por pregunta\n    votos_imp = df_votaciones[df_votaciones['tipo_voto']=='impacto'].groupby('pregunta_critica_id').size().rename('votos_impacto')\n    votos_urg = df_votaciones[df_votaciones['tipo_voto']=='urgencia'].groupby('pregunta_critica_id').size().rename('votos_urgencia')\n    num_fricc = df_fricciones.groupby('pregunta_critica_id').size().rename('num_fricciones')\n    \n    df_score = df_preguntas.copy()\n    df_score = df_score.merge(votos_imp, left_on='id', right_index=True, how='left')\n    df_score = df_score.merge(votos_urg, left_on='id', right_index=True, how='left')\n    df_score = df_score.merge(num_fricc, left_on='id', right_index=True, how='left')\n    df_score[['votos_impacto','votos_urgencia','num_fricciones']] = df_score[['votos_impacto','votos_urgencia','num_fricciones']].fillna(0)\n    \n    df_score['freq_score'] = df_score['frecuencia'].map(freq_scores).fillna(0)\n    df_score['imp_score'] = df_score['impacto'].map(imp_scores).fillna(0)\n    df_score['score'] = (df_score['votos_impacto'] * 3) + (df_score['votos_urgencia'] * 2) + df_score['freq_score'] + df_score['imp_score'] + (df_score['num_fricciones'] * 1.5)\n    \n    df_score = df_score.sort_values('score', ascending=False)\n    \n    # Mostrar Top 15\n    cols = ['pregunta_clave', 'decision', 'area', 'capa', 'frecuencia', 'impacto', 'votos_impacto', 'votos_urgencia', 'num_fricciones', 'score']\n    print('🏆 TOP 15 PREGUNTAS CRÍTICAS POR SCORE COMPUESTO')\n    print('=' * 80)\n    display(df_score[cols].head(15).reset_index(drop=True))\n    \n    # Gráfico de scoring\n    fig, ax = plt.subplots(figsize=(14, 7))\n    top = df_score.head(15)\n    bars = ax.barh(range(len(top)), top['score'].values, color=plt.cm.RdYlGn_r([x/top['score'].max() for x in top['score'].values]))\n    ax.set_yticks(range(len(top)))\n    ax.set_yticklabels([t[:60]+'...' if len(t)>60 else t for t in top['pregunta_clave'].values], fontsize=9)\n    ax.invert_yaxis()\n    ax.set_xlabel('Score Compuesto')\n    ax.set_title('Top 15 Preguntas Críticas por Score Compuesto', fontsize=14, fontweight='bold')\n    for i, v in enumerate(top['score'].values):\n        ax.text(v + 0.3, i, f'{v:.1f}', va='center', fontweight='bold', fontsize=9)\n    plt.tight_layout()\n    plt.show()\nelse:\n    print('⚠️ No hay preguntas para calcular scoring')

## 4. Matriz de Priorización 2×2 (Impacto vs Urgencia)

In [ ]:
# Matriz 2x2: Impacto vs Urgencia\nif len(df_preguntas) > 0 and 'df_score' in dir() and len(df_score) > 0:\n    voted = df_score[(df_score['votos_impacto'] > 0) | (df_score['votos_urgencia'] > 0)].copy()\n    if len(voted) > 0:\n        fig, ax = plt.subplots(figsize=(12, 8))\n        mid_x = voted['votos_impacto'].median()\n        mid_y = voted['votos_urgencia'].median()\n        ax.axhline(y=mid_y, color='gray', linestyle='--', alpha=0.5)\n        ax.axvline(x=mid_x, color='gray', linestyle='--', alpha=0.5)\n        \n        # Cuadrantes\n        ax.fill_between([mid_x, voted['votos_impacto'].max()*1.2], mid_y, voted['votos_urgencia'].max()*1.2, alpha=0.08, color='red', label='🔥 HACER YA')\n        ax.fill_between([0, mid_x], mid_y, voted['votos_urgencia'].max()*1.2, alpha=0.08, color='orange', label='📋 PLANIFICAR')\n        ax.fill_between([mid_x, voted['votos_impacto'].max()*1.2], 0, mid_y, alpha=0.08, color='blue', label='⚡ ESTRATÉGICO')\n        ax.fill_between([0, mid_x], 0, mid_y, alpha=0.08, color='gray', label='📌 MONITOREAR')\n        \n        areas_unique = voted['area'].unique()\n        colors = plt.cm.Set1([i/len(areas_unique) for i in range(len(areas_unique))])\n        area_color = dict(zip(areas_unique, colors))\n        \n        for _, row in voted.iterrows():\n            ax.scatter(row['votos_impacto'], row['votos_urgencia'], s=row['score']*15, \n                      c=[area_color[row['area']]], alpha=0.7, edgecolors='black', linewidth=0.5)\n            ax.annotate(row['pregunta_clave'][:35]+'...', (row['votos_impacto'], row['votos_urgencia']),\n                       fontsize=7, ha='center', va='bottom', xytext=(0,8), textcoords='offset points')\n        \n        ax.set_xlabel('Votos de Impacto →', fontsize=12, fontweight='bold')\n        ax.set_ylabel('Votos de Urgencia →', fontsize=12, fontweight='bold')\n        ax.set_title('Matriz de Priorización: Impacto vs Urgencia\\n(Tamaño = Score Compuesto)', fontsize=14, fontweight='bold')\n        ax.legend(loc='upper left', fontsize=9)\n        plt.tight_layout()\n        plt.show()\n    else:\n        print('⚠️ No hay preguntas con votos para generar la matriz')\nelse:\n    print('⚠️ Datos insuficientes para la matriz 2x2')

## 5. Análisis Cross-Funcional por Área

In [ ]:
# Análisis cross-funcional por área\nif len(df_decisiones) > 0:\n    cross = df_gerentes.groupby('area').agg(\n        gerentes=('id', 'nunique')\n    ).reset_index()\n    \n    dec_area = df_decisiones.groupby('area').agg(decisiones=('id', 'count')).reset_index()\n    preg_area = df_preguntas.groupby('area').agg(preguntas=('id', 'count')).reset_index()\n    fric_area = df_fricciones.groupby('area').agg(fricciones=('id', 'count')).reset_index()\n    \n    cross = cross.merge(dec_area, on='area', how='left')\n    cross = cross.merge(preg_area, on='area', how='left')\n    cross = cross.merge(fric_area, on='area', how='left')\n    cross = cross.fillna(0)\n    \n    fig, axes = plt.subplots(1, 2, figsize=(14, 6))\n    \n    x = range(len(cross))\n    w = 0.25\n    axes[0].bar([i-w for i in x], cross['decisiones'], w, label='Decisiones', color='#3b82f6')\n    axes[0].bar(x, cross['preguntas'], w, label='Preguntas', color='#10b981')\n    axes[0].bar([i+w for i in x], cross['fricciones'], w, label='Fricciones', color='#ef4444')\n    axes[0].set_xticks(x)\n    axes[0].set_xticklabels(cross['area'], rotation=45, ha='right', fontsize=9)\n    axes[0].legend()\n    axes[0].set_title('Actividad por Área', fontweight='bold')\n    \n    axes[1].barh(cross['area'], cross['fricciones'], color='#ef4444')\n    axes[1].set_xlabel('Cantidad de Fricciones')\n    axes[1].set_title('Fricciones por Área (Dolor de Datos)', fontweight='bold')\n    axes[1].invert_yaxis()\n    \n    plt.tight_layout()\n    plt.show()\n    \n    print('\\n📊 Resumen Cross-Funcional:')\n    display(cross.sort_values('fricciones', ascending=False))\nelse:\n    print('⚠️ No hay datos suficientes')

## 6. Clasificación de Fricciones con IA (OpenAI GPT-4o-mini)

In [ ]:
# Clasificación de fricciones con OpenAI\nfrom openai import OpenAI\n\nclient = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))\n\nif len(df_fricciones) > 0:\n    fricciones_text = '\\n'.join([\n        f\"{i+1}. Área: {r['area']} | Decisión: {r.get('decision','N/A')} | Pregunta: {r.get('pregunta_clave','N/A')} | Situación: {r['situacion_actual']} | Consecuencia: {r['consecuencia']}\"\n        for i, r in df_fricciones.iterrows()\n    ])\n    \n    response = client.chat.completions.create(\n        model='gpt-4o-mini',\n        messages=[{'role': 'user', 'content': f\"\"\"Eres un consultor experto en estrategia de datos. Clasifica cada fricción en UNA categoría:\n- DISPONIBILIDAD: El dato no existe\n- OPORTUNIDAD: Llega tarde\n- CALIDAD: Incorrecto o incompleto\n- ACCESIBILIDAD: Difícil de obtener (silos, Excel)\n- GRANULARIDAD: No al nivel de detalle necesario\n- INTEGRACIÓN: En múltiples sistemas sin conectar\n\nPara cada una sugiere: acción concreta, esfuerzo (Bajo/Medio/Alto), tipo de solución.\n\nFRICCIONES:\\n{fricciones_text}\n\nResponde SOLO JSON: [{\"id\": 1, \"categoria\": \"...\", \"accion\": \"...\", \"esfuerzo\": \"...\", \"tipo_solucion\": \"...\", \"insight\": \"...\"}]\"\"\"}],\n        temperature=0.3, max_tokens=4000\n    )\n    \n    import re\n    content = response.choices[0].message.content.strip()\n    json_match = re.search(r'\\[.*\\]', content, re.DOTALL)\n    clasificaciones = json.loads(json_match.group()) if json_match else json.loads(content)\n    \n    df_clasificacion = pd.DataFrame(clasificaciones)\n    print('✅ Fricciones clasificadas con IA')\n    print(f'\\n📊 Distribución por categoría:')\n    print(df_clasificacion['categoria'].value_counts().to_string())\n    \n    # Gráfico de categorías\n    fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n    cat_colors = {'DISPONIBILIDAD':'#ef4444','OPORTUNIDAD':'#f59e0b','CALIDAD':'#8b5cf6','ACCESIBILIDAD':'#3b82f6','GRANULARIDAD':'#06b6d4','INTEGRACIÓN':'#ec4899'}\n    cat_counts = df_clasificacion['categoria'].value_counts()\n    axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.0f%%', colors=[cat_colors.get(c,'#94a3b8') for c in cat_counts.index], startangle=90)\n    axes[0].set_title('Distribución de Fricciones por Categoría', fontweight='bold')\n    \n    if 'esfuerzo' in df_clasificacion.columns:\n        esf_counts = df_clasificacion['esfuerzo'].value_counts()\n        axes[1].bar(esf_counts.index, esf_counts.values, color=['#10b981','#f59e0b','#ef4444'][:len(esf_counts)])\n        axes[1].set_title('Distribución por Esfuerzo Estimado', fontweight='bold')\n    plt.tight_layout()\n    plt.show()\n    \n    display(df_clasificacion)\nelse:\n    print('⚠️ No hay fricciones para clasificar')

## 7. Data Roadmap - Plan de Acción

In [ ]:
# Generar Data Roadmap\nif len(df_preguntas) > 0 and 'df_score' in dir():\n    print('=' * 90)\n    print('📋 DATA ROADMAP - PLAN DE ACCIÓN PARA ESTRATEGIA ANALÍTICA DE MASSY GROUP')\n    print('=' * 90)\n    \n    top_preguntas = df_score.head(10)\n    for i, (_, row) in enumerate(top_preguntas.iterrows(), 1):\n        print(f'\\n🎯 PRIORIDAD #{i} | Score: {row[\"score\"]:.1f}')\n        print(f'   📊 Decisión: {row[\"decision\"]}')\n        print(f'   ❓ Pregunta: {row[\"pregunta_clave\"]}')\n        print(f'   👤 Área: {row[\"area\"]} | Capa: {row.get(\"capa\",\"N/A\")} | Frecuencia: {row[\"frecuencia\"]} | Impacto: {row[\"impacto\"]}')\n        print(f'   🗳️ Votos: Impacto={int(row[\"votos_impacto\"])} | Urgencia={int(row[\"votos_urgencia\"])} | Fricciones={int(row[\"num_fricciones\"])}')\n        \n        # Buscar fricciones asociadas\n        fricc_asociadas = df_fricciones[df_fricciones['pregunta_critica_id'] == row['id']]\n        if len(fricc_asociadas) > 0:\n            for _, f in fricc_asociadas.iterrows():\n                print(f'   ⚠️ Fricción: {f[\"situacion_actual\"]} → {f[\"consecuencia\"]}')\n        print(f'   {\"─\" * 70}')\n    \n    print('\\n\\n📊 RESUMEN EJECUTIVO:')\n    print(f'   Total preguntas críticas: {len(df_preguntas)}')\n    print(f'   Total fricciones identificadas: {len(df_fricciones)}')\n    print(f'   Áreas involucradas: {df_gerentes[\"area\"].nunique()}')\n    print(f'   Participantes: {len(df_gerentes)}')\n    if len(df_decisiones) > 0:\n        print(f'   Decisiones de alto impacto (Alto+Crítico): {len(df_decisiones[df_decisiones[\"impacto\"].isin([\"Alto\",\"Crítico\"])])}')\n        print(f'   Decisiones de alta frecuencia (Diaria+Semanal): {len(df_decisiones[df_decisiones[\"frecuencia\"].isin([\"Diaria\",\"Semanal\"])])}')\nelse:\n    print('⚠️ No hay datos suficientes para generar el roadmap')

## 8. Exportar Resultados a Excel

In [ ]:
# Exportar todo a Excel con múltiples hojas\noutput_file = 'resultados_taller_analitica.xlsx'\n\nwith pd.ExcelWriter(output_file, engine='openpyxl') as writer:\n    df_gerentes.to_excel(writer, sheet_name='Gerentes', index=False)\n    if len(df_decisiones) > 0:\n        df_decisiones.to_excel(writer, sheet_name='Decisiones', index=False)\n    if len(df_preguntas) > 0:\n        df_preguntas.to_excel(writer, sheet_name='Preguntas', index=False)\n    if len(df_fricciones) > 0:\n        df_fricciones.to_excel(writer, sheet_name='Fricciones', index=False)\n    if len(df_votaciones) > 0:\n        df_votaciones.to_excel(writer, sheet_name='Votaciones', index=False)\n    if 'df_score' in dir() and len(df_score) > 0:\n        cols = ['pregunta_clave','decision','area','capa','frecuencia','impacto','votos_impacto','votos_urgencia','num_fricciones','score']\n        df_score[cols].to_excel(writer, sheet_name='Scoring', index=False)\n    if 'df_clasificacion' in dir() and len(df_clasificacion) > 0:\n        df_clasificacion.to_excel(writer, sheet_name='Clasificacion_IA', index=False)\n\nprint(f'✅ Resultados exportados a: {output_file}')\nprint(f'📊 Hojas incluidas: Gerentes, Decisiones, Preguntas, Fricciones, Votaciones, Scoring, Clasificacion_IA')

In [ ]:
# Cerrar conexión a SQL Server\nconn.close()\nprint('✅ Conexión cerrada. Análisis completado.')

# 🔬 Laboratorio de Ciencia de Datos - Taller de Analítica Massy Group\n\n## Objetivo\nAnalizar los datos recopilados en los talleres de enfoque analítico para:\n1. Identificar las decisiones más críticas por área y capa organizacional\n2. Priorizar preguntas críticas usando scoring compuesto\n3. Clasificar fricciones de información con IA\n4. Generar un Data Roadmap accionable\n\n## Capas Organizacionales\n- **Estratégico**: C-Level y Gerentes\n- **Táctico**: Líderes de negocio\n- **Operativo**: Personal del día a día\n\n---